# Parcial 1 - Simulacion

## Criterios teoricos: tipos de simulacion

- **Simulacion numerica**: aproxima valores de un sistema con operaciones matematicas y repeticion computacional (por ejemplo, Montecarlo).
- **Eventos discretos**: el sistema cambia por eventos puntuales (llegadas, solicitudes, salidas). El modelo del ascensor es discreto.
- **Eventos continuos**: las variables evolucionan en el tiempo de forma progresiva (por ejemplo, concentracion de un farmaco).
- **Deterministico**: no hay azar; mismas entradas generan mismas salidas.
- **Estocastico**: incorpora variabilidad aleatoria; mismas entradas pueden dar resultados distintos.

En este parcial: 
- Punto 1: simulacion de eventos discretos y estocastica.
- Punto 2: simulacion de comportamiento continuo y estocastica (por variabilidad inter-paciente).

## Punto 1 - Espera del ascensor en edificio de 15 pisos

### Supuestos del modelo
- 15 pisos (1 al 15).
- El ascensor tarda 3 segundos por piso.
- Al solicitarlo, va directo al piso que hizo la solicitud.
- Sin horas pico.
- Estado del ascensor antes de la solicitud: piso 1 es 3 veces mas probable que cualquier otro piso del 2 al 15.

In [41]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

np.random.seed(2026)

N_FLOORS = 15
SECONDS_PER_FLOOR = 3
SIMULATIONS = 200_000

floors = np.arange(1, N_FLOORS + 1)
weights = np.ones(N_FLOORS)
weights[0] = 3
probabilities = weights / weights.sum()

pd.DataFrame({
    "Piso": floors,
    "Peso": weights,
    "Probabilidad": probabilities
})

,Piso,Peso,Probabilidad
0,1,3.0,0.176471
1,2,1.0,0.058824
2,3,1.0,0.058824
3,4,1.0,0.058824
4,5,1.0,0.058824
5,6,1.0,0.058824
6,7,1.0,0.058824
7,8,1.0,0.058824
8,9,1.0,0.058824
9,10,1.0,0.058824


In [42]:
def run_elevator_scenario(request_floor, n_simulations=SIMULATIONS, bins=50):
    initial_positions = np.random.choice(floors, size=n_simulations, p=probabilities)
    wait_times = np.abs(initial_positions - request_floor) * SECONDS_PER_FLOOR
    avg_wait = wait_times.mean()

    print(f"Escenario: solicitud desde piso {request_floor}")
    print(f"Tiempo de espera promedio: {avg_wait:.2f} segundos")

    hist_df = pd.DataFrame({"Espera (s)": wait_times})
    fig = px.histogram(
        hist_df,
        x="Espera (s)",
        nbins=bins,
        histnorm="probability",
        title=f"Distribucion del tiempo de espera (piso {request_floor})",
    )
    fig.update_layout(yaxis_title="Probabilidad")
    fig.show()

    return initial_positions, wait_times, avg_wait

In [43]:
positions_floor1, waits_floor1, avg_wait_floor1 = run_elevator_scenario(request_floor=1)

Escenario: solicitud desde piso 1
Tiempo de espera promedio: 18.53 segundos


In [44]:
positions_floor15, waits_floor15, avg_wait_floor15 = run_elevator_scenario(request_floor=15)

Escenario: solicitud desde piso 15
Tiempo de espera promedio: 23.46 segundos


### Analisis e interpretacion (Punto 1)

- Como el ascensor es mucho mas probable en el piso 1, la espera promedio al pedirlo desde el piso 1 tiende a ser baja.
- Desde el piso 15, la espera promedio aumenta porque la mayoria de posiciones iniciales estan lejos de ese piso.

### Que pasaria si...

Se evalua como cambia la espera promedio si: 
1. El ascensor tarda menos por piso (2 segundos).
2. Se agrega un segundo ascensor identico e independiente (el usuario toma el que llegue primero).

In [45]:
def wait_time_from_positions(initial_positions, request_floor, seconds_per_floor):
    return np.abs(initial_positions - request_floor) * seconds_per_floor

def simulate_two_elevators(request_floor, n_simulations=SIMULATIONS, seconds_per_floor=SECONDS_PER_FLOOR):
    pos_e1 = np.random.choice(floors, size=n_simulations, p=probabilities)
    pos_e2 = np.random.choice(floors, size=n_simulations, p=probabilities)
    w1 = wait_time_from_positions(pos_e1, request_floor, seconds_per_floor)
    w2 = wait_time_from_positions(pos_e2, request_floor, seconds_per_floor)
    return np.minimum(w1, w2)

base_floor1 = waits_floor1
faster_floor1 = wait_time_from_positions(positions_floor1, 1, seconds_per_floor=2)
two_elev_floor1 = simulate_two_elevators(request_floor=1)

scenario_df = pd.DataFrame({
    "Escenario": [
        "Base: 1 ascensor, 3 s/piso",
        "Mejora A: 1 ascensor, 2 s/piso",
        "Mejora B: 2 ascensores, 3 s/piso"
    ],
    "Espera promedio (s)": [
        base_floor1.mean(),
        faster_floor1.mean(),
        two_elev_floor1.mean()
    ]
})
scenario_df

,Escenario,Espera promedio (s)
0,"Base: 1 ascensor, 3 s/piso",18.528495
1,"Mejora A: 1 ascensor, 2 s/piso",12.352330
2,"Mejora B: 2 ascensores, 3 s/piso",10.544685


**Diagnostico sugerido:** si la espera objetivo no se cumple con un ascensor, la simulacion muestra el efecto de aumentar velocidad o instalar un segundo ascensor para reducir la cola de espera.

## Punto 2 - Nivel de droga con tasa de eliminacion normal

Se mantiene la idea del ejercicio de medicamento, pero ahora **cada paciente** tiene una tasa de eliminacion aleatoria con distribucion normal:
- Media = 0.55% por minuto = 0.0055
- Desviacion estandar = 0.15% por minuto = 0.0015

Esto modela variabilidad entre pacientes (modelo estocastico).

In [46]:
ELIMINATION_MEAN = 0.0055
ELIMINATION_STD = 0.0015
SIMULATION_TIME = 60 * 24
THERAPEUTIC_MIN = 200
THERAPEUTIC_MAX = 600
N_PATIENTS = 5000

DOSE_AMOUNT = 350
INITIAL_CONCENTRATION = 570
TIME_BETWEEN_DOSES = 60 * 3

def draw_patient_elimination_rates(n_patients):
    rates = np.random.normal(ELIMINATION_MEAN, ELIMINATION_STD, size=n_patients)
    return np.clip(rates, 0.0001, 0.03)

def simulate_patient_profile(initial_concentration, dose_amount, time_between_doses, elimination_rate, sim_time=SIMULATION_TIME):
    concentration = initial_concentration
    profile = np.empty(sim_time + 1)

    for t in range(sim_time + 1):
        profile[t] = concentration
        concentration *= (1 - elimination_rate)
        if t != 0 and t % time_between_doses == 0:
            concentration += dose_amount

    return profile

In [47]:
rates = draw_patient_elimination_rates(N_PATIENTS)
profiles = np.array([
    simulate_patient_profile(INITIAL_CONCENTRATION, DOSE_AMOUNT, TIME_BETWEEN_DOSES, r)
    for r in rates
])

mean_profile = profiles.mean(axis=0)
p10_profile = np.percentile(profiles, 10, axis=0)
p90_profile = np.percentile(profiles, 90, axis=0)

time_in_range = ((profiles >= THERAPEUTIC_MIN) & (profiles <= THERAPEUTIC_MAX)).mean(axis=1)

results_point2 = pd.DataFrame({
    "Metrica": [
        "Tasa eliminacion promedio simulada",
        "Desviacion tasa eliminacion simulada",
        "Porcentaje promedio de tiempo en rango terapeutico",
        "Pacientes con >= 80% del tiempo en rango"
    ],
    "Valor": [
        rates.mean(),
        rates.std(),
        100 * time_in_range.mean(),
        100 * np.mean(time_in_range >= 0.80)
    ]
})
results_point2

,Metrica,Valor
0,Tasa eliminacion promedio simulada,0.005535
1,Desviacion tasa eliminacion simulada,0.001496
2,Porcentaje promedio de tiempo en rango terapeu...,83.234171
3,Pacientes con >= 80% del tiempo en rango,64.900000


In [48]:
time_axis = np.arange(SIMULATION_TIME + 1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=time_axis, y=mean_profile, name="Promedio", line=dict(color="blue")))
fig.add_trace(go.Scatter(x=time_axis, y=p10_profile, name="Percentil 10", line=dict(color="gray", dash="dot")))
fig.add_trace(go.Scatter(x=time_axis, y=p90_profile, name="Percentil 90", line=dict(color="gray", dash="dot")))
fig.add_hline(y=THERAPEUTIC_MIN, line_dash="dash", line_color="red", annotation_text="Min terapeutico")
fig.add_hline(y=THERAPEUTIC_MAX, line_dash="dash", line_color="red", annotation_text="Max terapeutico")
fig.update_layout(
    title="Farmacocinetica Montecarlo con eliminacion normal",
    xaxis_title="Tiempo (min)",
    yaxis_title="Concentracion (mg/L)",
    hovermode="x unified"
)
fig.show()

### Analisis e interpretacion (Punto 2)

- Al pasar de una tasa fija (deterministica) a una tasa normal por paciente, se observa una banda de resultados, no una sola curva.
- Esto permite diagnosticar riesgo de subdosificacion o sobredosificacion en subgrupos.
- Si se desea mantener mas tiempo dentro de rango terapeutico, se pueden ajustar dosis o frecuencia.

### Que pasaria si...

- Si disminuye el intervalo entre dosis, suele subir la concentracion promedio y aumenta riesgo de superar maximo terapeutico.
- Si reduce la dosis por toma, suele disminuir toxicidad pero puede aumentar tiempo por debajo del minimo terapeutico.

Este tipo de simulacion soporta decisiones clinicas basadas en escenarios, no en un unico paciente promedio.